# 🎵 Audio Processing using Transformers — Complete Workflow

This notebook walks through every stage of a modern audio Transformer pipeline:

| Stage | Description |
|-------|-------------|
| 1 | Install & imports |
| 2 | Load raw audio |
| 3 | Pre-processing (resample, normalise, mono) |
| 4 | Feature extraction (Log-Mel spectrogram, MFCC, CQT) |
| 5 | Patch embedding + positional encoding |
| 6 | Transformer encoder (custom + HuggingFace) |
| 7 | Task heads (ASR, classification, generation) |
| 8 | Post-processing & evaluation |
| 9 | Fine-tuning with LoRA |
| 10 | Full end-to-end inference with Whisper |

---
> **Hardware**: GPU recommended (CUDA). CPU works for feature extraction; GPU needed for model inference.
>
> **Install**: run Stage 1 first.

---
## Stage 1 — Install dependencies

In [ ]:
# Core audio + ML libraries
!pip install -q librosa soundfile torchaudio
!pip install -q transformers datasets
!pip install -q torch torchvision
!pip install -q matplotlib numpy scipy
!pip install -q peft accelerate   # for LoRA fine-tuning
!pip install -q openai-whisper    # Whisper inference

print('All dependencies installed.')

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Audio, display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

import librosa
import librosa.display
import soundfile as sf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

---
## Stage 2 — Load raw audio

Audio is a 1-D signal of amplitude values sampled at a fixed rate.  
- `sample_rate` (Hz): how many samples per second  
- `waveform` shape: `[channels, num_samples]` (torchaudio) or `[num_samples]` (librosa)

In [ ]:
# ─── Option A: generate synthetic audio (sine wave + noise) ───
SAMPLE_RATE = 16000   # 16 kHz — standard for speech
DURATION    = 3.0     # seconds

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)

# Chord: 440 Hz (A4) + 550 Hz + 660 Hz (rough A major)
waveform_np = (
    0.4 * np.sin(2 * np.pi * 440 * t) +
    0.3 * np.sin(2 * np.pi * 550 * t) +
    0.2 * np.sin(2 * np.pi * 660 * t) +
    0.05 * np.random.randn(len(t))   # light background noise
).astype(np.float32)

# Convert to torch tensor: shape [1, T] (1 channel)
waveform = torch.from_numpy(waveform_np).unsqueeze(0)

print(f'Waveform shape : {waveform.shape}  (channels, samples)')
print(f'Sample rate    : {SAMPLE_RATE} Hz')
print(f'Duration       : {waveform.shape[1] / SAMPLE_RATE:.2f} s')
print(f'Amplitude range: [{waveform.min():.3f}, {waveform.max():.3f}]')

# Listen in notebook
display(Audio(waveform_np, rate=SAMPLE_RATE))

In [ ]:
# ─── Option B: load a real WAV file ───────────────────────────
# waveform, SAMPLE_RATE = torchaudio.load('your_file.wav')

# ─── Option C: load from HuggingFace dataset ──────────────────
# from datasets import load_dataset
# ds = load_dataset('speech_commands', 'v0.01', split='test')
# sample = ds[0]
# waveform_np = np.array(sample['audio']['array'], dtype=np.float32)
# SAMPLE_RATE  = sample['audio']['sampling_rate']
# waveform = torch.from_numpy(waveform_np).unsqueeze(0)

# ─── Visualise raw waveform ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 2.5))
times = np.linspace(0, waveform.shape[1] / SAMPLE_RATE, waveform.shape[1])
ax.plot(times, waveform[0].numpy(), linewidth=0.6, color='steelblue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Stage 2 — Raw waveform')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Stage 3 — Pre-processing

Three steps every audio pipeline needs before feature extraction:
1. **Resampling** — unify sample rate across dataset
2. **Normalisation** — stabilise amplitude range
3. **Mono downmix** — reduce stereo to single channel

In [ ]:
def preprocess_audio(
    waveform: torch.Tensor,
    orig_sr: int,
    target_sr: int = 16000,
    normalize: bool = True,
    to_mono: bool = True,
) -> torch.Tensor:
    """
    Full pre-processing pipeline.

    Parameters
    ----------
    waveform  : [C, T] float32 tensor
    orig_sr   : original sample rate
    target_sr : desired sample rate
    normalize : peak-normalise to [-1, 1]
    to_mono   : average channels → single channel

    Returns
    -------
    waveform  : [1, T'] pre-processed tensor
    """
    # Step 1 — Resample
    if orig_sr != target_sr:
        resampler = T.Resample(orig_freq=orig_sr, new_freq=target_sr)
        waveform  = resampler(waveform)
        print(f'  Resampled: {orig_sr} Hz → {target_sr} Hz')

    # Step 2 — Mono downmix
    if to_mono and waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
        print(f'  Downmixed to mono')

    # Step 3 — Peak normalisation
    if normalize:
        peak = waveform.abs().max()
        if peak > 0:
            waveform = waveform / peak
        print(f'  Normalised: peak={waveform.abs().max():.4f}')

    return waveform


# Apply to our waveform (already 16 kHz in this example)
print('Pre-processing steps:')
wav_clean = preprocess_audio(waveform, orig_sr=SAMPLE_RATE, target_sr=16000)

print(f'\nOutput shape: {wav_clean.shape}  (1, num_samples)')
print(f'Duration    : {wav_clean.shape[1] / 16000:.2f} s')

---
## Stage 4 — Feature Extraction

Transformers need a 2-D "image-like" input. We convert the 1-D waveform into a spectrogram:

```
Waveform [T]  →  STFT  →  Power spectrum [F, T_frames]  →  Mel filterbank  →  Log-Mel [n_mels, T_frames]
```

| Parameter | Typical value | Meaning |
|-----------|---------------|---------|
| `n_fft` | 400 | FFT window size (25 ms at 16 kHz) |
| `hop_length` | 160 | hop size (10 ms at 16 kHz) |
| `n_mels` | 80 | number of Mel frequency bins |
| `f_min` / `f_max` | 0 / 8000 Hz | frequency range |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4-A  Log-Mel Spectrogram (most common Transformer input)
# ═══════════════════════════════════════════════════════════════

N_FFT      = 400    # 25 ms window at 16 kHz
HOP_LENGTH = 160    # 10 ms hop
N_MELS     = 80     # Mel bins (Whisper uses 80)
F_MIN      = 0.0
F_MAX      = 8000.0

mel_transform = T.MelSpectrogram(
    sample_rate = 16000,
    n_fft       = N_FFT,
    hop_length  = HOP_LENGTH,
    n_mels      = N_MELS,
    f_min       = F_MIN,
    f_max       = F_MAX,
    power       = 2.0,          # power spectrogram
)

mel_spec = mel_transform(wav_clean)          # [1, n_mels, T_frames]
log_mel  = torch.log(mel_spec + 1e-9)        # log compression
log_mel  = log_mel.squeeze(0)                # [n_mels, T_frames]

print(f'Log-Mel shape: {log_mel.shape}  (n_mels={N_MELS}, T_frames={log_mel.shape[1]})')
print(f'T_frames ≈ {SAMPLE_RATE * DURATION / HOP_LENGTH:.0f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw spectrogram
img1 = axes[0].imshow(
    librosa.power_to_db(mel_spec.squeeze(0).numpy(), ref=np.max),
    aspect='auto', origin='lower', cmap='magma'
)
axes[0].set_title('Mel Spectrogram (dB)')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Mel bin')
plt.colorbar(img1, ax=axes[0], label='dB')

# Log-mel
img2 = axes[1].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Log-Mel Spectrogram')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Mel bin')
plt.colorbar(img2, ax=axes[1], label='log amplitude')

plt.suptitle('Stage 4-A — Log-Mel Spectrogram', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4-B  MFCC — classic speech feature
# ═══════════════════════════════════════════════════════════════
N_MFCC = 40

mfcc_transform = T.MFCC(
    sample_rate  = 16000,
    n_mfcc       = N_MFCC,
    melkwargs    = {'n_fft': N_FFT, 'hop_length': HOP_LENGTH, 'n_mels': 80},
)
mfcc = mfcc_transform(wav_clean).squeeze(0)  # [n_mfcc, T_frames]

# Compute delta and delta-delta (velocity & acceleration)
delta  = torchaudio.functional.compute_deltas(mfcc)
delta2 = torchaudio.functional.compute_deltas(delta)
mfcc_full = torch.cat([mfcc, delta, delta2], dim=0)  # [3*n_mfcc, T_frames]

print(f'MFCC shape          : {mfcc.shape}')
print(f'MFCC + delta + delta2: {mfcc_full.shape}')

# ═══════════════════════════════════════════════════════════════
# 4-C  CQT — constant-Q transform (music tasks)
# ═══════════════════════════════════════════════════════════════
wav_np = wav_clean.squeeze(0).numpy()
cqt    = np.abs(librosa.cqt(wav_np, sr=16000, hop_length=HOP_LENGTH,
                              fmin=librosa.note_to_hz('C2'), n_bins=84))
log_cqt = librosa.amplitude_to_db(cqt, ref=np.max)

print(f'CQT shape: {log_cqt.shape}  (bins, T_frames)')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].imshow(mfcc.numpy(), aspect='auto', origin='lower', cmap='coolwarm')
axes[0].set_title(f'MFCC ({N_MFCC} coefficients)')
axes[0].set_xlabel('Frame'); axes[0].set_ylabel('Coefficient')

axes[1].imshow(mfcc_full.numpy(), aspect='auto', origin='lower', cmap='coolwarm')
axes[1].set_title('MFCC + Δ + ΔΔ')
axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Feature')

axes[2].imshow(log_cqt, aspect='auto', origin='lower', cmap='inferno')
axes[2].set_title('CQT (music)')
axes[2].set_xlabel('Frame'); axes[2].set_ylabel('Frequency bin')

plt.suptitle('Stage 4-B/C — MFCC and CQT features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4-D  SpecAugment — training augmentation on spectrograms
# ═══════════════════════════════════════════════════════════════
freq_mask  = T.FrequencyMasking(freq_mask_param=15)
time_mask  = T.TimeMasking(time_mask_param=35)

spec_aug = log_mel.unsqueeze(0).clone()    # [1, n_mels, T_frames]
spec_aug = freq_mask(spec_aug)             # mask random frequency bands
spec_aug = time_mask(spec_aug)             # mask random time windows
spec_aug = spec_aug.squeeze(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original log-Mel')
axes[1].imshow(spec_aug.numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('After SpecAugment (freq + time masking)')
for ax in axes:
    ax.set_xlabel('Frame'); ax.set_ylabel('Mel bin')
plt.suptitle('Stage 4-D — SpecAugment data augmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 5 — Patch Embedding + Positional Encoding

The Transformer needs a sequence of fixed-size vectors.  
Each time-frame of the spectrogram is projected to `d_model` dimensions:

```
[T_frames, n_mels]  →  Linear(n_mels, d_model)  →  [T_frames, d_model]
                                                  + positional encoding
                                                  → [T_frames, d_model]
```

In [ ]:
class AudioPatchEmbedding(nn.Module):
    """
    Projects each spectrogram frame [n_mels] → [d_model].

    Whisper uses 2 × Conv1d instead — both approaches shown below.
    """
    def __init__(self, n_mels: int = 80, d_model: int = 256):
        super().__init__()
        self.linear = nn.Linear(n_mels, d_model)
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n_mels, T_frames]
        x = x.permute(0, 2, 1)           # → [batch, T_frames, n_mels]
        x = self.linear(x)               # → [batch, T_frames, d_model]
        return self.norm(x)


class WhisperStyleEmbedding(nn.Module):
    """
    Whisper-style: 2 × Conv1d before feeding to Transformer.
    """
    def __init__(self, n_mels: int = 80, d_model: int = 256):
        super().__init__()
        self.conv1 = nn.Conv1d(n_mels, d_model, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1)
        self.gelu  = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n_mels, T_frames]
        x = self.gelu(self.conv1(x))     # → [batch, d_model, T]
        x = self.gelu(self.conv2(x))     # → [batch, d_model, T//2]
        return x.permute(0, 2, 1)        # → [batch, T//2, d_model]


class SinusoidalPositionalEncoding(nn.Module):
    """
    Classic sinusoidal PE from 'Attention is All You Need'.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int = 256, max_len: int = 3000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, T, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Test the embedding pipeline ──────────────────────────────────
D_MODEL   = 256
BATCH     = 1

spec_input = log_mel.unsqueeze(0)  # [1, n_mels, T_frames]

embed   = AudioPatchEmbedding(n_mels=N_MELS, d_model=D_MODEL)
pos_enc = SinusoidalPositionalEncoding(d_model=D_MODEL)

embedded = embed(spec_input)       # [1, T_frames, 256]
encoded  = pos_enc(embedded)       # [1, T_frames, 256]

print(f'Spectrogram input  : {spec_input.shape}   (batch, n_mels, T_frames)')
print(f'After embedding    : {embedded.shape}   (batch, T_frames, d_model)')
print(f'After pos encoding : {encoded.shape}    (batch, T_frames, d_model)')

# Visualise positional encoding
pe_viz = pos_enc.pe[0, :100, :64].detach().numpy()
plt.figure(figsize=(12, 3))
plt.imshow(pe_viz.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(label='PE value')
plt.title('Stage 5 — Sinusoidal positional encoding (first 64 dims × 100 positions)')
plt.xlabel('Position (time frame)'); plt.ylabel('Dimension')
plt.tight_layout()
plt.show()

---
## Stage 6 — Transformer Encoder from scratch

Each encoder layer has:
1. **Multi-Head Self-Attention** — every frame attends to every other frame
2. **Add & LayerNorm** (residual)
3. **Feed-Forward Network** — 2-layer MLP with non-linearity
4. **Add & LayerNorm** (residual)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head self-attention.
    Attention(Q,K,V) = softmax(QKᵀ / √d_k) · V
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(self.d_k)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # [batch, T, d_model] → [batch, heads, T, d_k]
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor = None,
    ) -> tuple:
        Q = self.split_heads(self.W_q(x))   # [B, H, T, d_k]
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [B, H, T, T]
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)                     # [B, H, T, T]
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)                      # [B, H, T, d_k]
        context = context.transpose(1, 2).contiguous()               # [B, T, H, d_k]
        B, T, H, dk = context.shape
        context = context.view(B, T, H * dk)                         # [B, T, d_model]

        return self.W_o(context), attn_weights


class FeedForward(nn.Module):
    """Position-wise 2-layer MLP. d_ff = 4 × d_model."""
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x): return self.net(x)


class TransformerEncoderLayer(nn.Module):
    """Single encoder layer: MHSA → Add&Norm → FFN → Add&Norm."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        self.attn     = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff       = FeedForward(d_model, d_ff, dropout)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # MHSA with residual
        attn_out, attn_weights = self.attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # FFN with residual
        x = self.norm2(x + self.ff(x))
        return x, attn_weights


class AudioTransformerEncoder(nn.Module):
    """Full encoder: embedding → N × TransformerEncoderLayer."""
    def __init__(
        self,
        n_mels:   int = 80,
        d_model:  int = 256,
        n_heads:  int = 8,
        n_layers: int = 6,
        d_ff:     int = 1024,
        max_len:  int = 3000,
        dropout:  float = 0.1,
    ):
        super().__init__()
        self.embedding = AudioPatchEmbedding(n_mels, d_model)
        self.pos_enc   = SinusoidalPositionalEncoding(d_model, max_len, dropout)
        self.layers    = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.pos_enc(self.embedding(x))
        all_attn = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            all_attn.append(attn)
        return self.norm(x), all_attn


# ── Run a forward pass ──────────────────────────────────────────
encoder = AudioTransformerEncoder(
    n_mels=N_MELS, d_model=D_MODEL, n_heads=8, n_layers=4, d_ff=1024
)
encoder.eval()

with torch.no_grad():
    enc_out, attn_maps = encoder(spec_input)

print(f'Encoder input  : {spec_input.shape}')
print(f'Encoder output : {enc_out.shape}  (batch, T_frames, d_model)')
print(f'Attention maps : {len(attn_maps)} layers × {attn_maps[0].shape}')

n_params = sum(p.numel() for p in encoder.parameters())
print(f'Model params   : {n_params:,}')

In [ ]:
# ── Visualise attention maps ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Show first 4 heads from layer 0 and layer 3
for layer_idx, row in zip([0, 3], [0, 1]):
    for head in range(4):
        ax = axes[row][head]
        attn = attn_maps[layer_idx][0, head].detach().numpy()  # [T, T]
        im = ax.imshow(attn[:30, :30], cmap='Blues', aspect='auto', vmin=0)
        ax.set_title(f'Layer {layer_idx+1}, Head {head+1}', fontsize=9)
        ax.set_xlabel('Key frame', fontsize=8)
        ax.set_ylabel('Query frame', fontsize=8)

plt.suptitle('Stage 6 — Attention weight maps (first 30 frames)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 7 — Task-Specific Heads

The encoder output `[batch, T, d_model]` is fed into a task head:

| Task | Head type | Output |
|------|-----------|--------|
| Classification | Mean-pool + Linear | class logits |
| ASR (CTC) | Linear per frame + CTC decode | text tokens |
| Speaker ID | Mean-pool + cosine similarity | speaker embedding |
| Generation | Autoregressive Transformer decoder | audio/text tokens |

In [ ]:
# ════════════════════════════════════════════════════════════════
# 7-A  Audio Classification Head
# ════════════════════════════════════════════════════════════════
class AudioClassificationHead(nn.Module):
    """
    Mean-pools encoder frames → classifies into N classes.
    Used for: emotion, genre, keyword spotting, environment sounds.
    """
    def __init__(self, d_model: int, n_classes: int, dropout: float = 0.3):
        super().__init__()
        self.pool   = lambda x: x.mean(dim=1)   # [B, T, D] → [B, D]
        self.head   = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_classes),
        )

    def forward(self, enc_output: torch.Tensor):
        pooled = self.pool(enc_output)    # [B, d_model]
        logits = self.head(pooled)        # [B, n_classes]
        return logits, F.softmax(logits, dim=-1)


# ════════════════════════════════════════════════════════════════
# 7-B  CTC Head for ASR (Connectionist Temporal Classification)
# ════════════════════════════════════════════════════════════════
class CTCHead(nn.Module):
    """
    Projects each encoder frame to vocabulary logits.
    CTC allows framewise prediction → collapse blanks → text.
    """
    def __init__(self, d_model: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.proj    = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, enc_output: torch.Tensor):
        # enc_output: [B, T, d_model]
        logits = self.proj(self.dropout(enc_output))   # [B, T, vocab]
        log_probs = F.log_softmax(logits, dim=-1)
        return logits, log_probs


# ════════════════════════════════════════════════════════════════
# 7-C  Speaker Embedding Head
# ════════════════════════════════════════════════════════════════
class SpeakerEmbeddingHead(nn.Module):
    """Produces a fixed-dim speaker vector (d-vector / x-vector style)."""
    def __init__(self, d_model: int, embed_dim: int = 128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(d_model, embed_dim),
            nn.Tanh(),
        )

    def forward(self, enc_output: torch.Tensor):
        pooled = enc_output.mean(dim=1)    # [B, d_model]
        emb    = self.proj(pooled)         # [B, embed_dim]
        return F.normalize(emb, dim=-1)    # L2-normalised for cosine sim


# ── Demo forward passes ──────────────────────────────────────────
N_CLASSES  = 10  # e.g. 10 emotion categories
VOCAB_SIZE = 32  # e.g. 32 subword tokens

clf_head    = AudioClassificationHead(D_MODEL, N_CLASSES)
ctc_head    = CTCHead(D_MODEL, VOCAB_SIZE)
spk_head    = SpeakerEmbeddingHead(D_MODEL, embed_dim=128)

with torch.no_grad():
    clf_logits, clf_probs = clf_head(enc_out)
    ctc_logits, ctc_lp   = ctc_head(enc_out)
    spk_emb               = spk_head(enc_out)

print('Task head outputs:')
print(f'  Classification logits : {clf_logits.shape}  → class probs: {clf_probs.shape}')
print(f'  CTC log-probs         : {ctc_lp.shape}  (B, T_frames, vocab)')
print(f'  Speaker embedding     : {spk_emb.shape}  (B, embed_dim=128)')

# Visualise classification probabilities
classes = [f'Class {i}' for i in range(N_CLASSES)]
plt.figure(figsize=(9, 3))
plt.bar(classes, clf_probs[0].numpy(), color='steelblue')
plt.title('Stage 7-A — Classification probabilities (random weights)')
plt.ylabel('Probability')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Stage 8 — Training Loop with SpecAugment

Full training setup for audio classification. Key components:
- `AdamW` optimiser with warmup cosine schedule
- `SpecAugment` for regularisation
- Gradient clipping

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR


class AudioClassifier(nn.Module):
    """Full pipeline: spectrogram → encoder → classification head."""
    def __init__(self, n_mels, d_model, n_heads, n_layers, n_classes):
        super().__init__()
        self.mel = T.MelSpectrogram(16000, N_FFT, HOP_LENGTH, n_mels=n_mels)
        self.log = lambda x: torch.log(x + 1e-9)
        self.aug = nn.Sequential(
            T.FrequencyMasking(freq_mask_param=15),
            T.TimeMasking(time_mask_param=35),
        )
        self.enc  = AudioTransformerEncoder(n_mels, d_model, n_heads, n_layers)
        self.head = AudioClassificationHead(d_model, n_classes)

    def forward(self, wav, augment=True):
        spec = self.log(self.mel(wav))       # [B, n_mels, T]
        if augment and self.training:
            spec = self.aug(spec)
        enc_out, _ = self.enc(spec)          # [B, T, d_model]
        logits, probs = self.head(enc_out)
        return logits


def train_epoch(model, loader, optimiser, criterion, scaler, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for wav, labels in loader:
        wav, labels = wav.to(device), labels.to(device)
        optimiser.zero_grad()

        with torch.cuda.amp.autocast():          # mixed precision
            logits = model(wav, augment=True)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimiser)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        preds   = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


# ── Simulate one training step ───────────────────────────────────
model     = AudioClassifier(N_MELS, D_MODEL, n_heads=8, n_layers=4, n_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimiser = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

# Fake batch: 2 audio clips × 3 s
fake_wav    = torch.randn(2, 1, SAMPLE_RATE * 3).to(DEVICE)
fake_labels = torch.randint(0, 10, (2,)).to(DEVICE)

model.train()
optimiser.zero_grad()

logits = model(fake_wav, augment=True)
loss   = criterion(logits, fake_labels)
loss.backward()

torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimiser.step()

print(f'Simulated training step:')
print(f'  Input  : {fake_wav.shape}')
print(f'  Logits : {logits.shape}')
print(f'  Loss   : {loss.item():.4f}')
print(f'  Preds  : {logits.argmax(-1).tolist()}')
print(f'  Labels : {fake_labels.tolist()}')

---
## Stage 9 — Fine-tuning with HuggingFace + LoRA

Use a pre-trained Wav2Vec2 / Whisper and fine-tune efficiently with LoRA adapters.  
LoRA freezes base weights and trains tiny rank-decomposed matrices — only ~0.1–1% of parameters.

In [ ]:
# ── Load pre-trained Wav2Vec2 feature extractor ───────────────────
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

MODEL_ID   = 'facebook/wav2vec2-base'
N_CLASSES  = 10

print(f'Loading {MODEL_ID} …')
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

model_hf = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=N_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

total_params     = sum(p.numel() for p in model_hf.parameters())
trainable_params = sum(p.numel() for p in model_hf.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# ── Apply LoRA adapters ──────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type      = TaskType.AUDIO_CLASSIFICATION,
    r              = 8,           # rank — trade-off: expressivity vs params
    lora_alpha     = 16,          # scaling factor = alpha / r
    lora_dropout   = 0.05,
    target_modules = ['q_proj', 'v_proj'],  # apply LoRA to attention q + v
    bias           = 'none',
)

model_lora = get_peft_model(model_hf, lora_config)
model_lora.print_trainable_parameters()
# → trainable params: ~0.1–0.5% of total

# ── Pre-process audio for HuggingFace model ───────────────────────
inputs = feature_extractor(
    waveform_np,
    sampling_rate=16000,
    return_tensors='pt',
    padding=True,
    max_length=48000,        # 3 s × 16 kHz
    truncation=True,
)

model_lora.eval()
with torch.no_grad():
    outputs = model_lora(**{k: v.to(DEVICE) for k, v in inputs.items()})

logits = outputs.logits
probs  = logits.softmax(-1)

print(f'\nHuggingFace + LoRA inference:')
print(f'  Output logits : {logits.shape}')
print(f'  Predicted class: {probs.argmax(-1).item()} (prob={probs.max().item():.3f})')

---
## Stage 10 — End-to-End Inference with Whisper (ASR)

Whisper is a full encoder-decoder Transformer trained on 680k hours of multilingual audio.

Architecture summary:
- **Encoder**: 6 Transformer layers with 2 × Conv1d front-end
- **Decoder**: 6 autoregressive Transformer layers with cross-attention
- **Input**: 80-band log-Mel spectrogram
- **Output**: text tokens → decoded string

In [ ]:
from transformers import pipeline, WhisperProcessor, WhisperForConditionalGeneration

WHISPER_ID = 'openai/whisper-tiny'   # tiny=39M params; base=74M; small=244M

print(f'Loading {WHISPER_ID} …')
processor    = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID).to(DEVICE)
whisper_model.eval()

# Pre-process: waveform → 80-band log-Mel → input_features
input_features = processor(
    waveform_np,
    sampling_rate=16000,
    return_tensors='pt',
).input_features.to(DEVICE)

print(f'Input features shape : {input_features.shape}  (batch, 80 mels, 3000 frames)')

# Generate transcription with beam search
with torch.no_grad():
    predicted_ids = whisper_model.generate(
        input_features,
        language       = 'english',
        task           = 'transcribe',
        num_beams      = 5,
        max_new_tokens = 100,
    )

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print(f'\nTranscription: "{transcription[0]}"')
print('(Synthetic audio → likely empty or noise-based output)')

# Show model architecture summary
enc_params = sum(p.numel() for p in whisper_model.model.encoder.parameters())
dec_params = sum(p.numel() for p in whisper_model.model.decoder.parameters())
print(f'\nWhisper-tiny breakdown:')
print(f'  Encoder params : {enc_params:,}')
print(f'  Decoder params : {dec_params:,}')

In [ ]:
# ── Convenient pipeline API ──────────────────────────────────────
asr_pipe = pipeline(
    'automatic-speech-recognition',
    model     = WHISPER_ID,
    device    = 0 if DEVICE == 'cuda' else -1,
    chunk_length_s = 30,      # chunk long audio
    stride_length_s = [5, 5], # overlap for context
)

result = asr_pipe({'raw': waveform_np, 'sampling_rate': 16000})
print(f'ASR pipeline result: {result}')

---
## Stage 11 — Evaluation Metrics

| Task | Metric | Formula |
|------|--------|---------|
| ASR | WER (Word Error Rate) | (S+D+I) / N |
| ASR | CER (Character Error Rate) | char-level WER |
| Classification | Accuracy / F1 / AUC | standard |
| Generation | FID (audio), FAD, MOS | perceptual quality |

In [ ]:
def word_error_rate(reference: str, hypothesis: str) -> float:
    """
    Compute WER = (Substitutions + Deletions + Insertions) / N_reference_words.
    Uses dynamic programming (Wagner-Fischer algorithm).
    """
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    R, H      = len(ref_words), len(hyp_words)

    dp = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i
    for j in range(H + 1): dp[0][j] = j

    for i in range(1, R + 1):
        for j in range(1, H + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,        # deletion
                dp[i][j-1] + 1,        # insertion
                dp[i-1][j-1] + cost,   # substitution
            )

    return dp[R][H] / max(R, 1)


# Example
ref = "The quick brown fox jumps over the lazy dog"
hyp = "the quick brown fox jump over lazy dog"

wer = word_error_rate(ref, hyp)
print(f'Reference  : "{ref}"')
print(f'Hypothesis : "{hyp}"')
print(f'WER        : {wer:.2%}  ({wer:.4f})')

# Accuracy + confusion demo
true_labels = torch.tensor([0, 1, 2, 1, 0, 3, 2, 1, 0, 3])
pred_labels = torch.tensor([0, 1, 2, 1, 1, 3, 2, 0, 0, 3])
accuracy    = (true_labels == pred_labels).float().mean()
print(f'\nClassification accuracy: {accuracy:.2%}')

---
## Summary — Full pipeline recap

```
Raw audio [T samples]
   │
   ├─ Resample → 16 kHz
   ├─ Normalise → [-1, 1]
   └─ Mono downmix
         │
         ├─ STFT → Power spectrum [F, T_frames]
         ├─ Mel filterbank → [80, T_frames]
         └─ Log compression → log-Mel [80, T_frames]
               │
               ├─ Linear projection → [T_frames, d_model]
               └─ Positional encoding → [T_frames, d_model]
                     │
                     └─ N × TransformerEncoderLayer
                           ├─ MHSA: every frame ↔ every frame
                           ├─ Add & LayerNorm
                           ├─ FFN (d_ff = 4 × d_model)
                           └─ Add & LayerNorm
                                 │
                                 ├─ [ASR]         CTC / autoregressive decoder → text
                                 ├─ [clf]         mean-pool + Linear → class label
                                 ├─ [speaker ID]  mean-pool + L2-norm → embedding
                                 └─ [generation]  decoder + vocoder → waveform
```

### Key models reference

| Model | Year | Task | Params |
|-------|------|------|--------|
| wav2vec 2.0 | 2020 | ASR | 95M – 317M |
| HuBERT | 2021 | Self-supervised pre-training | 95M – 317M |
| AST | 2021 | Audio classification | 86M |
| Whisper | 2022 | ASR + translation | 39M – 1.5B |
| AudioLM | 2022 | Audio generation | — |
| MusicGen | 2023 | Music generation | 300M – 3.3B |
| Seamless | 2023 | Speech translation | — |
| Qwen-Audio | 2024 | Multimodal audio-LLM | 7B |